In [ ]:
import pandas as pd
from db_utils import get_db_connection, fetch_data

# Create database connection
conn = get_db_connection()

# Fetch data into pandas dataframe
raw_data = fetch_data(conn, "SELECT data, value FROM macro_data WHERE id = 'rate_gs10'")
derived_data = fetch_data(conn, "SELECT data, value FROM macro_data WHERE id IN ('r_sp_earn', 'r_sp_price', 'r_sp_div')")

# Fetch average future real earnings (rolling 1200-month forward average)
future_earnings = fetch_data(
    conn,
    """
    SELECT 
        date,
        AVG(value) OVER (
            ORDER BY date
            ROWS BETWEEN 1 FOLLOWING AND 1200 FOLLOWING
        ) as value
    FROM derived.shiller_cape
    WHERE id = 'real_earn'
    ORDER BY date
    """
)


# Close connection
conn.close()

# Print first 5 rows
print(data.head())

In [ ]:
# notebooks/predict_earnings.ipynb

import sys
import os
import pandas as pd

# Define the path to the directory containing 'db_utils'
# Adjust the relative path as necessary
current_dir = os.getcwd()
db_utils_path = os.path.join(current_dir, '..')  # Assuming 'db_utils' is one level up
db_utils_path = os.path.abspath(db_utils_path)

# Add 'db_utils' to sys.path if not already present
if db_utils_path not in sys.path:
    sys.path.insert(0, db_utils_path)

# Now, import the necessary functions
from db_utils import get_db_connection, fetch_data

# Create database connection
conn = get_db_connection()

# Fetch data into pandas dataframe
raw_data = fetch_data(conn, "SELECT date, id, value FROM macro_data WHERE id = 'rate_gs10'")
derived_data = fetch_data(conn, "SELECT date, id, value FROM shiller_derived_view WHERE id IN ('r_sp_earn', 'r_sp_price', 'r_sp_div')")

data = pd.concat([raw_data, derived_data], axis=0).set_index(['date', 'id'])['value'].unstack()

# Fetch average future real earnings (rolling 1200-month forward average)
future_earnings = fetch_data(
    conn,
    """
    SELECT 
        date,
        CASE
            WHEN COUNT(*) OVER (
                ORDER BY date 
                ROWS BETWEEN 1 FOLLOWING AND 120 FOLLOWING
            ) = 120 THEN
                AVG(value) OVER (
                    ORDER BY date
                    ROWS BETWEEN 1 FOLLOWING AND 120 FOLLOWING
                )
            ELSE NULL
        END as value
    FROM shiller_derived_view
    WHERE id = 'r_sp_earn'
    ORDER BY date
    """
).set_index('date').rename(columns={'value': 'future_earnings'})

# Close connection
conn.close()

# Print first 5 rows
print(data.head())

In [ ]:
future_earnings

In [ ]:
y = future_earnings.div(data.loc(1)['r_sp_price'], axis=0).dropna()['future_earnings'].rename('rel_future_earnings')

In [ ]:
y.plot()

In [ ]:
type(data.loc(1)['rate_gs10'])